# Comparative Analysis (With or Withour Uncertainty Estimation for GPT-2 Model)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np
import pandas as pd
import json
from scipy.special import softmax

train_path = "/content/drive/MyDrive/NIPS25/DATA/data_clean/data_clean/questions/US/train.jsonl"  # Update with the correct file path

def load_jsonl(file_path):
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

# Load training data
train_data = load_jsonl(train_path)

# Extract questions, options, and answers
questions = [item["question"] for item in train_data]
choices = [item["options"] for item in train_data]  # Use "options" instead of "choices"
answers = [item["answer"] for item in train_data]

# Load pre-trained GPT model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # Explicitly set padding token
model = AutoModelForCausalLM.from_pretrained("gpt2")

# Function to generate response without uncertainty modeling
def generate_response_no_uncertainty(input_text):
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    output_sequences = model.generate(inputs["input_ids"], max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(output_sequences[0], skip_special_tokens=True)
    return response

# Bayesian Calibration using Monte Carlo Dropout
class BayesianGPT(nn.Module):
    def __init__(self, base_model):
        super(BayesianGPT, self).__init__()
        self.base_model = base_model
        self.dropout = nn.Dropout(p=0.3)  # MC Dropout for Bayesian Estimation

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids, attention_mask=attention_mask)
        logits = self.dropout(outputs.logits)  # Apply dropout multiple times during inference
        return logits

# Apply Bayesian Model
bayesian_model = BayesianGPT(model)
bayesian_model.train()

# Function for uncertainty estimation
def estimate_uncertainty_gpt(input_text, n_samples=20):
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    input_ids, attention_mask = inputs["input_ids"], inputs["attention_mask"]
    predictions = []

    for _ in range(n_samples):
        with torch.no_grad():
            logits = bayesian_model(input_ids, attention_mask)
            predictions.append(logits.numpy())

    predictions = np.array(predictions)
    mean_prediction = predictions.mean(axis=0)
    uncertainty = np.abs(predictions.std(axis=0) / (predictions.mean(axis=0) + 1e-6))  # Ensure non-negative uncertainty

    return mean_prediction, uncertainty

# Clinical Uncertainty Score (CUS)
def compute_cus(uncertainties):
    return np.mean(uncertainties)

# Improved Zero-Shot Trustworthiness Index (ZTI)
def compute_zti(cus_scores, epsilon=1e-6):
    if len(cus_scores) < 2:
        return 1.0  # Default to fully trustworthy if only one sample is available

    mean_cus = np.mean(cus_scores)
    std_cus = np.std(cus_scores)
    max_cus = max(np.max(cus_scores), epsilon)  # Ensure non-zero max for stability

    zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))  # Ensure non-negative log values
    return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# Compute CUS and ZTI across multiple samples with and without uncertainty modeling
cus_scores_no_uncertainty = []
cus_scores_with_uncertainty = []
num_samples = min(10, len(questions))  # Evaluate on 10 questions for better stability

for i in range(num_samples):
    # Without uncertainty modeling
    response_no_uncertainty = generate_response_no_uncertainty(questions[i])
    print(f"Sample {i+1} - GPT Response (No Uncertainty): {response_no_uncertainty}")
    pred_no_uncertainty = model(tokenizer(questions[i], return_tensors="pt")["input_ids"])  # Raw prediction
    cus_score_no_uncertainty = compute_cus(pred_no_uncertainty.logits.detach().numpy())
    cus_scores_no_uncertainty.append(cus_score_no_uncertainty)
    print(f"Sample {i+1} - CUS (No Uncertainty): {cus_score_no_uncertainty}")

    # With uncertainty modeling
    pred, uncertainty = estimate_uncertainty_gpt(questions[i])
    cus_score = compute_cus(uncertainty)
    cus_scores_with_uncertainty.append(cus_score)
    print(f"Sample {i+1} - CUS (With Uncertainty): {cus_score}")

# Compute overall ZTI
zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) With Uncertainty:", zti_score_with_uncertainty)

# Generate final comparison report
comparison_results = {
    "CUS_No_Uncertainty": cus_scores_no_uncertainty,
    "CUS_With_Uncertainty": cus_scores_with_uncertainty,
    "ZTI_No_Uncertainty": zti_score_no_uncertainty,
    "ZTI_With_Uncertainty": zti_score_with_uncertainty,
}
print("Comparison Results:", comparison_results)


Sample 1 - GPT Response (No Uncertainty): A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?

The patient has a history of vomiting, or has a history of vomiting, or has a history of, or has a history of, or has a history of, or has a history of, or has a history of, or has a
Sample 1 - CUS (No Uncertainty): -106.37493133544922
Sample 1 - CUS (With Uncertainty): 2.7150871753692627
Sample 2 - GPT Response (No Uncertainty): A 3-month-old baby died suddenly at night

# Comparative Analysis (With or Withour Uncertainty Estimation for BERT Model)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np
import pandas as pd
import json
from scipy.special import softmax

train_path = "/content/drive/MyDrive/NIPS25/DATA/data_clean/data_clean/questions/US/train.jsonl"  # Update with the correct file path

def load_jsonl(file_path):
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

# Load training data
train_data = load_jsonl(train_path)

# Extract questions, options, and answers
questions = [item["question"] for item in train_data]
choices = [item["options"] for item in train_data]  # Use "options" instead of "choices"
answers = [item["answer"] for item in train_data]

# Load pre-trained BERT model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)

# Function to generate predictions without uncertainty modeling
def predict_no_uncertainty(input_text):
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    probs = softmax(logits.numpy(), axis=1)
    return probs

# Bayesian Calibration using Monte Carlo Dropout
class BayesianBERT(nn.Module):
    def __init__(self, base_model):
        super(BayesianBERT, self).__init__()
        self.base_model = base_model
        self.dropout = nn.Dropout(p=0.3)  # MC Dropout for Bayesian Estimation

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids, attention_mask=attention_mask)
        logits = self.dropout(outputs.logits)  # Apply dropout multiple times during inference
        return logits

# Apply Bayesian Model
bayesian_model = BayesianBERT(model)
bayesian_model.train()

# Function for uncertainty estimation
def estimate_uncertainty_bert(input_text, n_samples=20):
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    input_ids, attention_mask = inputs["input_ids"], inputs["attention_mask"]
    predictions = []

    for _ in range(n_samples):
        with torch.no_grad():
            logits = bayesian_model(input_ids, attention_mask)
            predictions.append(logits.numpy())

    predictions = np.array(predictions)
    mean_prediction = predictions.mean(axis=0)
    uncertainty = np.abs(predictions.std(axis=0) / (predictions.mean(axis=0) + 1e-6))  # Ensure non-negative uncertainty

    return mean_prediction, uncertainty

# Clinical Uncertainty Score (CUS)
def compute_cus(uncertainties):
    return np.mean(uncertainties)

# Improved Zero-Shot Trustworthiness Index (ZTI)
def compute_zti(cus_scores, epsilon=1e-6):
    if len(cus_scores) < 2:
        return 1.0  # Default to fully trustworthy if only one sample is available

    mean_cus = np.mean(cus_scores)
    std_cus = np.std(cus_scores)
    max_cus = max(np.max(cus_scores), epsilon)  # Ensure non-zero max for stability

    zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))  # Ensure non-negative log values
    return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# Compute CUS and ZTI across multiple samples with and without uncertainty modeling
cus_scores_no_uncertainty = []
cus_scores_with_uncertainty = []
num_samples = min(10, len(questions))  # Evaluate on 10 questions for better stability

for i in range(num_samples):
    # Without uncertainty modeling
    probs_no_uncertainty = predict_no_uncertainty(questions[i])
    cus_score_no_uncertainty = compute_cus(probs_no_uncertainty)
    cus_scores_no_uncertainty.append(cus_score_no_uncertainty)
    print(f"Sample {i+1} - CUS (No Uncertainty): {cus_score_no_uncertainty}")

    # With uncertainty modeling
    pred, uncertainty = estimate_uncertainty_bert(questions[i])
    cus_score = compute_cus(uncertainty)
    cus_scores_with_uncertainty.append(cus_score)
    print(f"Sample {i+1} - CUS (With Uncertainty): {cus_score}")

# Compute overall ZTI
zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) With Uncertainty:", zti_score_with_uncertainty)

# Generate final comparison report
comparison_results = {
    "CUS_No_Uncertainty": cus_scores_no_uncertainty,
    "CUS_With_Uncertainty": cus_scores_with_uncertainty,
    "ZTI_No_Uncertainty": zti_score_no_uncertainty,
    "ZTI_With_Uncertainty": zti_score_with_uncertainty,
}
print("Comparison Results:", comparison_results)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Sample 1 - CUS (No Uncertainty): 0.2499999850988388
Sample 1 - CUS (With Uncertainty): 1.7053723335266113
Sample 2 - CUS (No Uncertainty): 0.25
Sample 2 - CUS (With Uncertainty): 0.9546841979026794
Sample 3 - CUS (No Uncertainty): 0.25
Sample 3 - CUS (With Uncertainty): 1.1495267152786255
Sample 4 - CUS (No Uncertainty): 0.25
Sample 4 - CUS (With Uncertainty): 0.7580061554908752
Sample 5 - CUS (No Uncertainty): 0.25
Sample 5 - CUS (With Uncertainty): 1.055035948753357
Sample 6 - CUS (No Uncertainty): 0.25
Sample 6 - CUS (With Uncertainty): 2.1543712615966797
Sample 7 - CUS (No Uncertainty): 0.2499999850988388
Sample 7 - CUS (With Uncertainty): 8.68504524230957
Sample 8 - CUS (No Uncertainty): 0.25
Sample 8 - CUS (With Uncertainty): 1.1512336730957031
Sample 9 - CUS (No Uncertainty): 0.25
Sample 9 - CUS (With Uncertainty): 1.465058445930481
Sample 10 - CUS (No Uncertainty): 0.25
Sample 10 - CUS (With Uncertainty): 1.2148321866989136
Zero-Shot Trustworthiness Index (ZTI) No Uncertainty: 

# Comparative Analysis (With or Withour Uncertainty Estimation for gpt-neo-1.3B Model)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np
import pandas as pd
import json
from scipy.special import softmax

train_path = "/content/drive/MyDrive/NIPS25/DATA/data_clean/data_clean/questions/US/train.jsonl"  # Update with the correct file path

def load_jsonl(file_path):
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

# Load training data
train_data = load_jsonl(train_path)

# Extract questions, options, and answers
questions = [item["question"] for item in train_data]
choices = [item["options"] for item in train_data]  # Use "options" instead of "choices"
answers = [item["answer"] for item in train_data]

# Extract questions and answers
questions = [item["question"] for item in train_data]
answers = [item["answer"] for item in train_data]

# Load pre-trained GPT-3 model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-1.3B")
tokenizer.pad_token = tokenizer.eos_token  # Explicitly set padding token
model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-1.3B")

# Function to generate response without uncertainty modeling
def generate_response_no_uncertainty(input_text):
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    output_sequences = model.generate(inputs["input_ids"], max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(output_sequences[0], skip_special_tokens=True)
    return response

# Bayesian Calibration using Monte Carlo Dropout
class BayesianGPT(nn.Module):
    def __init__(self, base_model):
        super(BayesianGPT, self).__init__()
        self.base_model = base_model
        self.dropout = nn.Dropout(p=0.5)  # Increased dropout rate for better uncertainty estimation

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids, attention_mask=attention_mask)
        logits = self.dropout(outputs.logits)  # Apply dropout multiple times during inference
        return logits

# Apply Bayesian Model
bayesian_model = BayesianGPT(model)
bayesian_model.train()

# Function for uncertainty estimation
def estimate_uncertainty_gpt(input_text, n_samples=20):
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    input_ids, attention_mask = inputs["input_ids"], inputs["attention_mask"]
    predictions = []

    for _ in range(n_samples):
        with torch.no_grad():
            logits = bayesian_model(input_ids, attention_mask)
            predictions.append(softmax(logits.numpy(), axis=-1))  # Normalize logits using softmax

    predictions = np.array(predictions)
    mean_prediction = predictions.mean(axis=0)
    uncertainty = np.abs(predictions.std(axis=0) / (mean_prediction + 1e-6))  # Normalized uncertainty

    return mean_prediction, uncertainty

# Clinical Uncertainty Score (CUS)
def compute_cus(uncertainties):
    return np.mean(uncertainties)

# Improved Zero-Shot Trustworthiness Index (ZTI)
def compute_zti(cus_scores, epsilon=1e-6):
    if len(cus_scores) < 2:
        return 1.0  # Default to fully trustworthy if only one sample is available

    mean_cus = np.mean(cus_scores)
    std_cus = np.std(cus_scores)
    max_cus = max(np.max(cus_scores), epsilon)  # Ensure non-zero max for stability

    zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))  # Ensure non-negative log values
    return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# Compute CUS and ZTI across multiple samples with and without uncertainty modeling
cus_scores_no_uncertainty = []
cus_scores_with_uncertainty = []
num_samples = min(10, len(questions))  # Evaluate on 10 questions for better stability

for i in range(num_samples):
    # Without uncertainty modeling
    response_no_uncertainty = generate_response_no_uncertainty(questions[i])
    print(f"Sample {i+1} - GPT-3 Response (No Uncertainty): {response_no_uncertainty}")
    pred_no_uncertainty = model(tokenizer(questions[i], return_tensors="pt", padding=True, truncation=True)["input_ids"])  # Raw prediction
    cus_score_no_uncertainty = compute_cus(softmax(pred_no_uncertainty.logits.detach().numpy(), axis=-1))  # Normalize logits
    cus_scores_no_uncertainty.append(cus_score_no_uncertainty)
    print(f"Sample {i+1} - CUS (No Uncertainty): {cus_score_no_uncertainty}")

    # With uncertainty modeling
    pred, uncertainty = estimate_uncertainty_gpt(questions[i])
    cus_score = compute_cus(uncertainty)
    cus_scores_with_uncertainty.append(cus_score)
    print(f"Sample {i+1} - CUS (With Uncertainty): {cus_score}")

# Compute overall ZTI
zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) With Uncertainty:", zti_score_with_uncertainty)

# Generate final comparison report
comparison_results = {
    "CUS_No_Uncertainty": cus_scores_no_uncertainty,
    "CUS_With_Uncertainty": cus_scores_with_uncertainty,
    "ZTI_No_Uncertainty": zti_score_no_uncertainty,
    "ZTI_With_Uncertainty": zti_score_with_uncertainty,
}
print("Comparison Results:", comparison_results)


Sample 1 - GPT-3 Response (No Uncertainty): A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?

A. Antibiotics

B. Antihistamines

C. Anticonvulsants

D. Antihistamines and anticonvulsants

E. Antihistamines and anticonv
Sample 1 - CUS (No Uncertainty): 1.989773409150075e-05
Sample 1 - CUS (With Uncertainty): 0.980335533618927
Sample 2 - GPT-3 Response (No Uncertainty): A 3-month-old baby died suddenly at night while asleep. His mother noticed that he had died on

# OpenAI GPT-3 Comparision

In [ ]:
import openai
import numpy as np
import json
import time
import torch
import torch.nn as nn
from openai import OpenAI

# Set your OpenAI API key
client = OpenAI(api_key="REDACTED_USE_ENV_OPENAI_API_KEY")

# Load dataset
train_path = "/content/drive/MyDrive/NIPS25/DATA/data_clean/data_clean/questions/US/train.jsonl"

def load_jsonl(file_path):
    """Load dataset from a JSONL file."""
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

# Load training data
train_data = load_jsonl(train_path)
questions = [item["question"] for item in train_data]

# **Simulated Bayesian Dropout with GPT-3.5 API Calls**
class BayesianGPT3_5(nn.Module):
    def __init__(self):
        super(BayesianGPT3_5, self).__init__()
        self.dropout = nn.Dropout(p=0.3)  # Apply MC Dropout (simulated via multiple API calls)

    def forward(self, input_text, n_samples=10):
        """Simulate Bayesian inference with multiple API calls (Monte Carlo Estimation)."""
        predictions = []

        for _ in range(n_samples):
            response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": input_text}],
                max_tokens=50,
                temperature=np.random.uniform(0.7, 1.2)  # Randomized temp for diversity
            )
            predictions.append(response.choices[0].message.content)
            time.sleep(1)  # Avoid rate limit

        # Convert predictions to numerical values
        prediction_lengths = [len(pred.split()) for pred in predictions]
        mean_prediction = np.mean(prediction_lengths)
        uncertainty = np.abs(np.std(prediction_lengths) / (mean_prediction + 1e-6))  # Normalize uncertainty

        return mean_prediction, uncertainty

# Instantiate Bayesian GPT-3.5 Model
bayesian_model = BayesianGPT3_5()

# Generate response without uncertainty modeling
def generate_response_no_uncertainty(input_text):
    """Generate response using GPT-3.5 without uncertainty modeling."""
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": input_text}],
        max_tokens=50,
        temperature=1.0
    )
    return response.choices[0].message.content

# Clinical Uncertainty Score (CUS)
def compute_cus(uncertainties):
    """Compute the Clinical Uncertainty Score (CUS)."""
    return np.mean(uncertainties)

# Improved Zero-Shot Trustworthiness Index (ZTI)
def compute_zti(cus_scores, epsilon=1e-6):
    """Compute Zero-Shot Trustworthiness Index (ZTI)."""
    if len(cus_scores) < 2:
        return 1.0  # Default to fully trustworthy if only one sample is available

    mean_cus = np.mean(cus_scores)
    std_cus = np.std(cus_scores)
    max_cus = max(np.max(cus_scores), epsilon)  # Ensure non-zero max for stability

    zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))  # Logarithmic scaling
    return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# Compute CUS and ZTI across multiple samples with and without uncertainty modeling
cus_scores_no_uncertainty = []
cus_scores_with_uncertainty = []
num_samples = min(10, len(questions))  # Evaluate on 10 questions

for i in range(num_samples):
    # Without uncertainty modeling
    response_no_uncertainty = generate_response_no_uncertainty(questions[i])
    print(f"Sample {i+1} - GPT-3.5 Response (No Uncertainty): {response_no_uncertainty}")
    cus_score_no_uncertainty = len(response_no_uncertainty.split())  # Length-based uncertainty proxy
    cus_scores_no_uncertainty.append(cus_score_no_uncertainty)
    print(f"Sample {i+1} - CUS (No Uncertainty): {cus_score_no_uncertainty}")

    # With Bayesian Uncertainty Modeling
    mean_pred, uncertainty = bayesian_model(questions[i], n_samples=10)
    cus_score = compute_cus([uncertainty])
    cus_scores_with_uncertainty.append(cus_score)
    print(f"Sample {i+1} - CUS (With Uncertainty): {cus_score}")

# Compute overall ZTI
zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) With Uncertainty:", zti_score_with_uncertainty)

# Generate final comparison report
comparison_results = {
    "CUS_No_Uncertainty": cus_scores_no_uncertainty,
    "CUS_With_Uncertainty": cus_scores_with_uncertainty,
    "ZTI_No_Uncertainty": zti_score_no_uncertainty,
    "ZTI_With_Uncertainty": zti_score_with_uncertainty,
}
print("Comparison Results:", comparison_results)


Sample 1 - GPT-3.5 Response (No Uncertainty): The best treatment for a pregnant woman with a urinary tract infection is a course of antibiotics. In this case, the most appropriate antibiotics for UTI in pregnancy are nitrofurantoin, cephalexin, or amoxicillin-clavulan
Sample 1 - CUS (No Uncertainty): 33
Sample 1 - CUS (With Uncertainty): 0.061167772727744124
Sample 2 - GPT-3.5 Response (No Uncertainty): 1. Placing the baby on his back to sleep: The American Academy of Pediatrics recommends that babies should always be placed on their backs to sleep to reduce the risk of Sudden Infant Death Syndrome (SIDS).

2. Using a firm sleep surface
Sample 2 - CUS (No Uncertainty): 42
Sample 2 - CUS (With Uncertainty): 0.03277462694214259
Sample 3 - GPT-3.5 Response (No Uncertainty): The presentation of a 3-week-old infant with regurgitation of feeds, yellow vomit, and abdominal distension is suggestive of pyloric stenosis. Pyloric stenosis is a condition where there is narrowing of the pyloric
Sam

# OpenAI GPT-4 Comparision

In [ ]:
import openai
import numpy as np
import json
import time
import torch
import torch.nn as nn
from openai import OpenAI

# Set your OpenAI API key
client = OpenAI(api_key="REDACTED_USE_ENV_OPENAI_API_KEY")  # Set your API key

# Load dataset
train_path = "/content/drive/MyDrive/NIPS25/DATA/data_clean/data_clean/questions/US/train.jsonl"

def load_jsonl(file_path):
    """Load dataset from a JSONL file."""
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

# Load training data
train_data = load_jsonl(train_path)
questions = [item["question"] for item in train_data]

# **Simulated Bayesian Dropout with GPT-4 API Calls**
class BayesianGPT4(nn.Module):
    def __init__(self):
        super(BayesianGPT4, self).__init__()
        self.dropout = nn.Dropout(p=0.3)  # Apply MC Dropout (simulated via multiple API calls)

    def forward(self, input_text, n_samples=10):
        """Simulate Bayesian inference with multiple API calls (Monte Carlo Estimation)."""
        predictions = []

        for _ in range(n_samples):
            response = client.chat.completions.create(
                model="gpt-4",
                messages=[{"role": "user", "content": input_text}],
                max_tokens=50,
                temperature=np.random.uniform(0.7, 1.2)  # Randomized temp for diversity
            )
            predictions.append(response.choices[0].message.content)
            time.sleep(1)  # Avoid rate limit

        # Convert predictions to numerical values
        prediction_lengths = [len(pred.split()) for pred in predictions]
        mean_prediction = np.mean(prediction_lengths)
        uncertainty = np.abs(np.std(prediction_lengths) / (mean_prediction + 1e-6))  # Normalize uncertainty

        return mean_prediction, uncertainty

# Instantiate Bayesian GPT-4 Model
bayesian_model = BayesianGPT4()

# Generate response without uncertainty modeling
def generate_response_no_uncertainty(input_text):
    """Generate response using GPT-4 without uncertainty modeling."""
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": input_text}],
        max_tokens=50,
        temperature=1.0
    )
    return response.choices[0].message.content

# Clinical Uncertainty Score (CUS)
def compute_cus(uncertainties):
    """Compute the Clinical Uncertainty Score (CUS)."""
    return np.mean(uncertainties)

# Improved Zero-Shot Trustworthiness Index (ZTI)
def compute_zti(cus_scores, epsilon=1e-6):
    """Compute Zero-Shot Trustworthiness Index (ZTI)."""
    if len(cus_scores) < 2:
        return 1.0  # Default to fully trustworthy if only one sample is available

    mean_cus = np.mean(cus_scores)
    std_cus = np.std(cus_scores)
    max_cus = max(np.max(cus_scores), epsilon)  # Ensure non-zero max for stability

    zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))  # Logarithmic scaling
    return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# Compute CUS and ZTI across multiple samples with and without uncertainty modeling
cus_scores_no_uncertainty = []
cus_scores_with_uncertainty = []
num_samples = min(10, len(questions))  # Evaluate on 10 questions

for i in range(num_samples):
    # Without uncertainty modeling
    response_no_uncertainty = generate_response_no_uncertainty(questions[i])
    print(f"Sample {i+1} - GPT-4 Response (No Uncertainty): {response_no_uncertainty}")
    cus_score_no_uncertainty = len(response_no_uncertainty.split())  # Length-based uncertainty proxy
    cus_scores_no_uncertainty.append(cus_score_no_uncertainty)
    print(f"Sample {i+1} - CUS (No Uncertainty): {cus_score_no_uncertainty}")

    # With Bayesian Uncertainty Modeling
    mean_pred, uncertainty = bayesian_model(questions[i], n_samples=10)
    cus_score = compute_cus([uncertainty])
    cus_scores_with_uncertainty.append(cus_score)
    print(f"Sample {i+1} - CUS (With Uncertainty): {cus_score}")

# Compute overall ZTI
zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) With Uncertainty:", zti_score_with_uncertainty)

# Generate final comparison report
comparison_results = {
    "CUS_No_Uncertainty": cus_scores_no_uncertainty,
    "CUS_With_Uncertainty": cus_scores_with_uncertainty,
    "ZTI_No_Uncertainty": zti_score_no_uncertainty,
    "ZTI_With_Uncertainty": zti_score_with_uncertainty,
}
print("Comparison Results:", comparison_results)


Sample 1 - GPT-4 Response (No Uncertainty): The best treatment for this patient is Nitrofurantoin. The patient's symptoms of dysuria are suggestive of a urinary tract infection (UTI), which are more common in pregnancy due to urinary stasis from ureteral dilation and compression of
Sample 1 - CUS (No Uncertainty): 37
Sample 1 - CUS (With Uncertainty): 0.04514469368040548
Sample 2 - GPT-4 Response (No Uncertainty): It's challenging to suggest specific precautions that could have prevented such a tragic event, especially when there's no determined cause of death based on the autopsy. That could indicate a case of Sudden Infant Death Syndrome (SIDS), though it's important to
Sample 2 - CUS (No Uncertainty): 41
Sample 2 - CUS (With Uncertainty): 0.041508125939092255
Sample 3 - GPT-4 Response (No Uncertainty): The symptoms described in this infant could be suggestive of pyloric stenosis. This condition presents in the third to fifth week of life with regurgitation and projectile nonbilious 

# GPT 4.5 and 4o model on MedQA

In [1]:
import openai
import numpy as np
import json
import time
import torch
import torch.nn as nn
from openai import OpenAI

# ✅ Set OpenAI API key
client = OpenAI(api_key="REDACTED_USE_ENV_OPENAI_API_KEY")

# ✅ Load MedQA dataset from JSONL
train_path = "/content/drive/MyDrive/NIPS25/DATA/data_clean/data_clean/questions/US/train.jsonl"

def load_jsonl(file_path):
    """Load dataset from a JSONL file."""
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

# Load training data
train_data = load_jsonl(train_path)
questions = [item["question"] for item in train_data]
num_samples = min(10, len(questions))  # Limit to 10 samples for evaluation

# ✅ Define Bayesian GPT Model for Uncertainty Estimation
class BayesianGPT(nn.Module):
    def __init__(self, model_name):
        super(BayesianGPT, self).__init__()
        self.model_name = model_name
        self.dropout = nn.Dropout(p=0.3)  # Apply MC Dropout (simulated via multiple API calls)

    def forward(self, input_text, n_samples=10):
        """Simulate Bayesian inference with multiple API calls (Monte Carlo Estimation)."""
        predictions = []

        for _ in range(n_samples):
            response = client.chat.completions.create(
                model=self.model_name,
                messages=[{"role": "user", "content": input_text}],
                max_tokens=50,
                temperature=np.random.uniform(0.7, 1.2)  # Randomized temp for diversity
            )
            predictions.append(response.choices[0].message.content)
            time.sleep(1)  # Avoid rate limit

        # Convert predictions to numerical values
        prediction_lengths = [len(pred.split()) for pred in predictions]
        mean_prediction = np.mean(prediction_lengths)
        uncertainty = np.abs(np.std(prediction_lengths) / (mean_prediction + 1e-6))  # Normalize uncertainty

        return mean_prediction, uncertainty

# ✅ Instantiate Bayesian Models for GPT-4.5 and GPT-4o
bayesian_gpt_45 = BayesianGPT("gpt-4.5-preview")
bayesian_gpt_4o = BayesianGPT("gpt-4o")

# ✅ Function: Generate Response Without Uncertainty
def generate_response_no_uncertainty(input_text, model_name):
    """Generate response using GPT model without uncertainty modeling."""
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": input_text}],
        max_tokens=50,
        temperature=1.0
    )
    return response.choices[0].message.content

# ✅ Clinical Uncertainty Score (CUS)
def compute_cus(uncertainties):
    """Compute the Clinical Uncertainty Score (CUS)."""
    return np.mean(uncertainties)

# ✅ Improved Zero-Shot Trustworthiness Index (ZTI)
def compute_zti(cus_scores, epsilon=1e-6):
    """Compute Zero-Shot Trustworthiness Index (ZTI)."""
    if len(cus_scores) < 2:
        return 1.0  # Default to fully trustworthy if only one sample is available

    mean_cus = np.mean(cus_scores)
    std_cus = np.std(cus_scores)
    max_cus = max(np.max(cus_scores), epsilon)  # Ensure non-zero max for stability

    zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))  # Logarithmic scaling
    return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# ✅ Evaluate GPT-4.5 and GPT-4o on MedQA dataset
models = {
    "GPT-4.5": bayesian_gpt_45,
    "GPT-4o": bayesian_gpt_4o
}

results = {}

for model_name, model in models.items():
    print(f"\n====== Running {model_name} on MedQA ======")

    cus_scores_no_uncertainty = []
    cus_scores_with_uncertainty = []

    for i in range(num_samples):
        # 🔹 Without Uncertainty
        response_no_uncertainty = generate_response_no_uncertainty(questions[i], model.model_name)
        print(f"Sample {i+1} - {model_name} Response (No Uncertainty): {response_no_uncertainty}")

        cus_score_no_uncertainty = len(response_no_uncertainty.split())  # Length-based uncertainty proxy
        cus_scores_no_uncertainty.append(cus_score_no_uncertainty)
        print(f"Sample {i+1} - {model_name} CUS (No Uncertainty): {cus_score_no_uncertainty}")

        # 🔹 With Bayesian Uncertainty Modeling
        mean_pred, uncertainty = model(questions[i], n_samples=10)
        cus_score = compute_cus([uncertainty])
        cus_scores_with_uncertainty.append(cus_score)
        print(f"Sample {i+1} - {model_name} CUS (With Uncertainty): {cus_score}")

    # ✅ Compute Overall ZTI
    zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
    zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)
    print(f"{model_name} Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
    print(f"{model_name} Zero-Shot Trustworthiness Index (ZTI) With Uncertainty:", zti_score_with_uncertainty)

    # ✅ Store results
    results[model_name] = {
        "CUS_No_Uncertainty": cus_scores_no_uncertainty,
        "CUS_With_Uncertainty": cus_scores_with_uncertainty,
        "ZTI_No_Uncertainty": zti_score_no_uncertainty,
        "ZTI_With_Uncertainty": zti_score_with_uncertainty,
    }

# ✅ Print final comparison report
print("\n===== Final Comparison Report =====")
for model_name, metrics in results.items():
    print(f"\nModel: {model_name}")
    print("CUS (No Uncertainty):", metrics["CUS_No_Uncertainty"])
    print("CUS (With Uncertainty):", metrics["CUS_With_Uncertainty"])
    print("ZTI (No Uncertainty):", metrics["ZTI_No_Uncertainty"])
    print("ZTI (With Uncertainty):", metrics["ZTI_With_Uncertainty"])



====== Running GPT-4.5 on MedQA ======
Sample 1 - GPT-4.5 Response (No Uncertainty): The correct answer in this clinical scenario is oral antibiotics—specifically nitrofurantoin, amoxicillin-clavulanate, or cephalexin.

Explanation:

This patient is presenting with symptoms typical of an acute uncomplicated urinary tract infection (
Sample 1 - GPT-4.5 CUS (No Uncertainty): 30
Sample 1 - GPT-4.5 CUS (With Uncertainty): 0.1232690769702446
Sample 2 - GPT-4.5 Response (No Uncertainty): You have not provided the list of possible answers. However, based on the clinical scenario you described—a sudden unexpected death of an apparently healthy infant under one year of age, usually during sleep, with no identifiable cause even after thorough investigation—this scenario strongly
Sample 2 - GPT-4.5 CUS (No Uncertainty): 43
Sample 2 - GPT-4.5 CUS (With Uncertainty): 0.1147078639166529
Sample 3 - GPT-4.5 Response (No Uncertainty): **Answer: Failure of gut rotation and fixation**

This newborn infa

# LLaMA-2 7B Comparision

In [ ]:
! huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `newtoken` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `newtoken`

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import json
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load LLaMA-2 7B model
MODEL_NAME = "meta-llama/Llama-2-7b-hf"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # Fix: Set padding token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")

# Enable dropout for Bayesian Approximation (Monte Carlo)
class BayesianLLaMA(nn.Module):
    def __init__(self, base_model):
        super(BayesianLLaMA, self).__init__()
        self.base_model = base_model
        self.dropout = nn.Dropout(p=0.3)  # Apply MC Dropout

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        logits = self.dropout(outputs.logits)  # Apply dropout at logits level
        return logits

# Wrap the model with Bayesian uncertainty modeling
bayesian_model = BayesianLLaMA(model)
bayesian_model.train()  # Ensure dropout is active

# Load dataset
train_path = "/content/drive/MyDrive/NIPS25/DATA/data_clean/data_clean/questions/US/train.jsonl"

def load_jsonl(file_path):
    """Load dataset from a JSONL file."""
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

# Load training data
train_data = load_jsonl(train_path)
questions = [item["question"] for item in train_data]

# Generate response without uncertainty modeling
def generate_response_no_uncertainty(input_text):
    """Generate response using LLaMA-2 without uncertainty modeling."""
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512, pad_to_multiple_of=8).to("cuda")
    with torch.no_grad():
        output_sequences = model.generate(inputs["input_ids"], max_new_tokens=50, pad_token_id=tokenizer.pad_token_id)
    response = tokenizer.decode(output_sequences[0], skip_special_tokens=True)
    return response

# Function for Bayesian Uncertainty Estimation
def estimate_uncertainty_llama(input_text, n_samples=10):
    """Estimate uncertainty using Bayesian Monte Carlo Dropout."""
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512, pad_to_multiple_of=8).to("cuda")
    predictions = []

    for _ in range(n_samples):
        with torch.no_grad():
            logits = bayesian_model(inputs["input_ids"], inputs["attention_mask"])
            predictions.append(logits.cpu().numpy())

    predictions = np.array(predictions)
    mean_prediction = predictions.mean(axis=0)
    uncertainty = np.abs(predictions.std(axis=0) / (mean_prediction + 1e-6))  # Normalize uncertainty

    return mean_prediction, uncertainty

# Clinical Uncertainty Score (CUS)
def compute_cus(uncertainties):
    """Compute Clinical Uncertainty Score (CUS)."""
    return np.mean(uncertainties)

# Zero-Shot Trustworthiness Index (ZTI)
def compute_zti(cus_scores, epsilon=1e-6):
    """Compute Zero-Shot Trustworthiness Index (ZTI)."""
    if len(cus_scores) < 2:
        return 1.0  # Default to fully trustworthy if only one sample is available

    mean_cus = np.mean(cus_scores)
    std_cus = np.std(cus_scores)
    max_cus = max(np.max(cus_scores), epsilon)  # Ensure non-zero max for stability

    zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))  # Logarithmic scaling
    return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# Compute CUS and ZTI across multiple samples
cus_scores_no_uncertainty = []
cus_scores_with_uncertainty = []
num_samples = min(10, len(questions))

for i in range(num_samples):
    # Without uncertainty modeling
    response_no_uncertainty = generate_response_no_uncertainty(questions[i])
    print(f"Sample {i+1} - LLaMA-2 7B Response (No Uncertainty): {response_no_uncertainty}")
    cus_score_no_uncertainty = len(response_no_uncertainty.split())  # Length-based proxy
    cus_scores_no_uncertainty.append(cus_score_no_uncertainty)
    print(f"Sample {i+1} - CUS (No Uncertainty): {cus_score_no_uncertainty}")

    # With Bayesian uncertainty modeling
    mean_pred, uncertainty = estimate_uncertainty_llama(questions[i])
    cus_score = compute_cus([uncertainty])
    cus_scores_with_uncertainty.append(cus_score)
    print(f"Sample {i+1} - CUS (With Uncertainty): {cus_score}")

# Compute ZTI scores
zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) With Uncertainty:", zti_score_with_uncertainty)

# Final Comparison Report
comparison_results = {
    "CUS_No_Uncertainty": cus_scores_no_uncertainty,
    "CUS_With_Uncertainty": cus_scores_with_uncertainty,
    "ZTI_No_Uncertainty": zti_score_no_uncertainty,
    "ZTI_With_Uncertainty": zti_score_with_uncertainty,
}
print("Comparison Results:", comparison_results)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Sample 1 - LLaMA-2 7B Response (No Uncertainty): A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient? Cool, dry air from the northwest will bring cooler temperatures to the region through the weekend.
The National Weather Service has issued a freeze warning for the North State from 10 p.m. Tues
Sample 1 - CUS (No Uncertainty): 128
Sample 1 - CUS (With Uncertainty): 0.66015625
Sample 2 - LLaMA-2 7B Response (No Uncertainty): A 3-month-old baby died suddenly at night

<ipython-input-9-f2b4490be13c>:66: RuntimeWarning: invalid value encountered in divide
  uncertainty = np.abs(predictions.std(axis=0) / (mean_prediction + 1e-6))  # Normalize uncertainty


Sample 9 - CUS (With Uncertainty): nan
Sample 10 - LLaMA-2 7B Response (No Uncertainty): A 35-year-old male presents to his primary care physician with complaints of seasonal allergies. He has been using intranasal vasoconstrictors several times per day for several weeks. What is a likely sequela of the chronic use of topical nasal decongestants? The National Center for State Courts (NCSC) is a nonprofit court improvement organization serving all 50 states, the federal judiciary and the territories. NCSC is the only organization in the nation dedicated solely to
Sample 10 - CUS (No Uncertainty): 74
Sample 10 - CUS (With Uncertainty): 0.66064453125
Zero-Shot Trustworthiness Index (ZTI) No Uncertainty: 0.13362852093812227
Zero-Shot Trustworthiness Index (ZTI) With Uncertainty: 0
Comparison Results: {'CUS_No_Uncertainty': [128, 79, 119, 140, 120, 181, 167, 86, 156, 74], 'CUS_With_Uncertainty': [0.66, 0.66, 0.66, 0.66, 0.66, 0.66, 0.66, 0.66, nan, 0.6606], 'ZTI_No_Uncertainty': 0.133628520

# LLaMA-2 13B Comparision

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import json
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load LLaMA-2 13B model
MODEL_NAME = "meta-llama/Llama-2-13b-hf"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # Fix padding token issue
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")

# Enable dropout for Bayesian Approximation (Monte Carlo)
class BayesianLLaMA(nn.Module):
    def __init__(self, base_model):
        super(BayesianLLaMA, self).__init__()
        self.base_model = base_model
        self.dropout = nn.Dropout(p=0.3)  # Apply MC Dropout

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        logits = self.dropout(outputs.logits)  # Apply dropout at logits level
        return logits

# Wrap the model with Bayesian uncertainty modeling
bayesian_model = BayesianLLaMA(model)
bayesian_model.train()  # Ensure dropout is active

# Load dataset
train_path = "/content/drive/MyDrive/NIPS25/DATA/data_clean/data_clean/questions/US/train.jsonl"

def load_jsonl(file_path):
    """Load dataset from a JSONL file."""
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

# Load training data
train_data = load_jsonl(train_path)
questions = [item["question"] for item in train_data]

# Generate response without uncertainty modeling
def generate_response_no_uncertainty(input_text):
    """Generate response using LLaMA-2 13B without uncertainty modeling."""
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512, pad_to_multiple_of=8).to("cuda")
    with torch.no_grad():
        output_sequences = model.generate(inputs["input_ids"], max_new_tokens=50, pad_token_id=tokenizer.pad_token_id)
    response = tokenizer.decode(output_sequences[0], skip_special_tokens=True)
    return response

# Function for Bayesian Uncertainty Estimation
def estimate_uncertainty_llama(input_text, n_samples=10):
    """Estimate uncertainty using Bayesian Monte Carlo Dropout."""
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512, pad_to_multiple_of=8).to("cuda")
    predictions = []

    for _ in range(n_samples):
        with torch.no_grad():
            logits = bayesian_model(inputs["input_ids"], inputs["attention_mask"])
            predictions.append(logits.cpu().numpy())

    predictions = np.array(predictions)
    mean_prediction = predictions.mean(axis=0)
    uncertainty = np.abs(predictions.std(axis=0) / (mean_prediction + 1e-6))  # Normalize uncertainty

    return mean_prediction, uncertainty

# Clinical Uncertainty Score (CUS)
def compute_cus(uncertainties):
    """Compute Clinical Uncertainty Score (CUS)."""
    return np.mean(uncertainties)

# Zero-Shot Trustworthiness Index (ZTI)
def compute_zti(cus_scores, epsilon=1e-6):
    """Compute Zero-Shot Trustworthiness Index (ZTI)."""
    if len(cus_scores) < 2:
        return 1.0  # Default to fully trustworthy if only one sample is available

    mean_cus = np.mean(cus_scores)
    std_cus = np.std(cus_scores)
    max_cus = max(np.max(cus_scores), epsilon)  # Ensure non-zero max for stability

    zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))  # Logarithmic scaling
    return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# Compute CUS and ZTI across multiple samples
cus_scores_no_uncertainty = []
cus_scores_with_uncertainty = []
num_samples = min(10, len(questions))

for i in range(num_samples):
    # Without uncertainty modeling
    response_no_uncertainty = generate_response_no_uncertainty(questions[i])
    print(f"Sample {i+1} - LLaMA-2 13B Response (No Uncertainty): {response_no_uncertainty}")
    cus_score_no_uncertainty = len(response_no_uncertainty.split())  # Length-based proxy
    cus_scores_no_uncertainty.append(cus_score_no_uncertainty)
    print(f"Sample {i+1} - CUS (No Uncertainty): {cus_score_no_uncertainty}")

    # With Bayesian uncertainty modeling
    mean_pred, uncertainty = estimate_uncertainty_llama(questions[i])
    cus_score = compute_cus([uncertainty])
    cus_scores_with_uncertainty.append(cus_score)
    print(f"Sample {i+1} - CUS (With Uncertainty): {cus_score}")

# Compute ZTI scores
zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) With Uncertainty:", zti_score_with_uncertainty)

# Final Comparison Report
comparison_results = {
    "CUS_No_Uncertainty": cus_scores_no_uncertainty,
    "CUS_With_Uncertainty": cus_scores_with_uncertainty,
    "ZTI_No_Uncertainty": zti_score_no_uncertainty,
    "ZTI_With_Uncertainty": zti_score_with_uncertainty,
}
print("Comparison Results:", comparison_results)


tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/610 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/33.4k [00:00<?, ?B/s]

... (2 outputs left out)

model-00003-of-00003.safetensors:   0%|          | 0.00/6.18G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

Sample 1 - LLaMA-2 13B Response (No Uncertainty): A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient? The Best of 2015: The Most Popular Posts on the Blog
The year 2015 is coming to an end. This year I have been sharing a lot of stuff on the blog and I thought it
Sample 1 - CUS (No Uncertainty): 131
Sample 1 - CUS (With Uncertainty): 0.66015625
Sample 2 - LLaMA-2 13B Response (No Uncertainty): A 3-month-old baby died suddenly at night while asleep. His mother noti

# BERT-Large, BioBERT, ClinicalBERT, and RoBERTa

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax

# Define model names for different BERT variants
BERT_MODELS = {
    "BERT-Large": "bert-large-uncased",
    "BioBERT": "dmis-lab/biobert-base-cased-v1.1",
    "ClinicalBERT": "emilyalsentzer/Bio_ClinicalBERT",
    "RoBERTa": "roberta-base"
}

# Load dataset
train_path = "/content/drive/MyDrive/NIPS25/DATA/data_clean/data_clean/questions/US/train.jsonl"

def load_jsonl(file_path):
    """Load dataset from a JSONL file."""
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

# Load training data
train_data = load_jsonl(train_path)
questions = [item["question"] for item in train_data]

# Define Bayesian Calibration using Monte Carlo Dropout
class BayesianBERT(nn.Module):
    def __init__(self, base_model):
        super(BayesianBERT, self).__init__()
        self.base_model = base_model
        self.dropout = nn.Dropout(p=0.3)  # MC Dropout for Bayesian Estimation

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids, attention_mask=attention_mask)
        logits = self.dropout(outputs.logits)  # Apply dropout multiple times during inference
        return logits

# Function to generate predictions without uncertainty modeling
def predict_no_uncertainty(model, tokenizer, input_text):
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    probs = softmax(logits.numpy(), axis=1)
    return probs

# Function for Bayesian Uncertainty Estimation
def estimate_uncertainty_bert(model, tokenizer, input_text, n_samples=20):
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    input_ids, attention_mask = inputs["input_ids"], inputs["attention_mask"]
    predictions = []

    for _ in range(n_samples):
        with torch.no_grad():
            logits = model(input_ids, attention_mask)
            predictions.append(logits.numpy())

    predictions = np.array(predictions)
    mean_prediction = predictions.mean(axis=0)
    uncertainty = np.abs(predictions.std(axis=0) / (predictions.mean(axis=0) + 1e-6))  # Ensure non-negative uncertainty

    return mean_prediction, uncertainty

# Clinical Uncertainty Score (CUS)
def compute_cus(uncertainties):
    return np.mean(uncertainties)

# Improved Zero-Shot Trustworthiness Index (ZTI)
def compute_zti(cus_scores, epsilon=1e-6):
    if len(cus_scores) < 2:
        return 1.0  # Default to fully trustworthy if only one sample is available

    mean_cus = np.mean(cus_scores)
    std_cus = np.std(cus_scores)
    max_cus = max(np.max(cus_scores), epsilon)  # Ensure non-zero max for stability

    zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))  # Ensure non-negative log values
    return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# Iterate over different BERT models
results = {}
num_samples = min(10, len(questions))  # Evaluate on 10 questions for better stability

for model_name, model_path in BERT_MODELS.items():
    print(f"\n====== Running {model_name} ======")

    # Load pre-trained model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=4)

    # Apply Bayesian Model
    bayesian_model = BayesianBERT(model)
    bayesian_model.train()

    cus_scores_no_uncertainty = []
    cus_scores_with_uncertainty = []

    for i in range(num_samples):
        # Without uncertainty modeling
        probs_no_uncertainty = predict_no_uncertainty(model, tokenizer, questions[i])
        cus_score_no_uncertainty = compute_cus(probs_no_uncertainty)
        cus_scores_no_uncertainty.append(cus_score_no_uncertainty)
        print(f"Sample {i+1} - {model_name} CUS (No Uncertainty): {cus_score_no_uncertainty}")

        # With uncertainty modeling
        pred, uncertainty = estimate_uncertainty_bert(bayesian_model, tokenizer, questions[i])
        cus_score = compute_cus(uncertainty)
        cus_scores_with_uncertainty.append(cus_score)
        print(f"Sample {i+1} - {model_name} CUS (With Uncertainty): {cus_score}")

    # Compute overall ZTI
    zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
    zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)
    print(f"{model_name} Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
    print(f"{model_name} Zero-Shot Trustworthiness Index (ZTI) With Uncertainty:", zti_score_with_uncertainty)

    # Store results
    results[model_name] = {
        "CUS_No_Uncertainty": cus_scores_no_uncertainty,
        "CUS_With_Uncertainty": cus_scores_with_uncertainty,
        "ZTI_No_Uncertainty": zti_score_no_uncertainty,
        "ZTI_With_Uncertainty": zti_score_with_uncertainty,
    }

# Print final comparison report
print("\n===== Final Comparison Report =====")
for model_name, metrics in results.items():
    print(f"\nModel: {model_name}")
    print("CUS (No Uncertainty):", metrics["CUS_No_Uncertainty"])
    print("CUS (With Uncertainty):", metrics["CUS_With_Uncertainty"])
    print("ZTI (No Uncertainty):", metrics["ZTI_No_Uncertainty"])
    print("ZTI (With Uncertainty):", metrics["ZTI_With_Uncertainty"])



====== Running BERT-Large ======


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

... (21 outputs left out)

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Sample 1 - RoBERTa CUS (No Uncertainty): 0.25
Sample 1 - RoBERTa CUS (With Uncertainty): 2.155836582183838
Sample 2 - RoBERTa CUS (No Uncertainty): 0.25
Sample 2 - RoBERTa CUS (With Uncertainty): 9.55683422088623
Sample 3 - RoBERTa CUS (No Uncertainty): 0.2499999850988388
Sample 3 - RoBERTa CUS (With Uncertainty): 1.8321534395217896
Sample 4 - RoBERTa CUS (No Uncertainty): 0.2499999850988388
Sample 4 - RoBERTa CUS (With Uncertainty): 1.571338415145874
Sample 5 - RoBERTa CUS (No Uncertainty): 0.25
Sample 5 - RoBERTa CUS (With Uncertainty): 3.3581717014312744
Sample 6 - RoBERTa CUS (No Uncertainty): 0.25
Sample 6 - RoBERTa CUS (With Uncertainty): 10.339628219604492
Sample 7 - RoBERTa CUS (No Uncertainty): 0.25
Sample 7 - RoBERTa CUS (With Uncertainty): 1.8809618949890137
Sample 8 - RoBERTa CUS (No Uncertainty): 0.2499999850988388
Sample 8 - RoBERTa CUS (With Uncertainty): 1.760769248008728
Sample 9 - RoBERTa CUS (No Uncertainty): 0.2499999850988388
Sample 9 - RoBERTa CUS (With Uncertaint